# SAE "Analyse automatique de données" -- Notebook 3 : classification multiclasse

Dans ce notebook, vous devez :
1. Effectuer des expérimentations pour optimiser les paramètres du classifieur choisi (k plus proche voisin ou arbre de décision) pour effectuer une classification multiclasse des avis ;
2. [Bonus] Effectuer des expérimentations supplémentaires avec d'autres classifieurs (l'autre classifieur vu en cours ou d'autres classifieurs de `scikit-learn`) ;
3. Rédiger une synthèse pour justifier des pré-traitements effectués, du choix du regroupement des notes pour obtenir vos classes et des valeurs des paramètres obtenus pour vos classifieurs. Vous concluerez sur le meilleur choix des pré-traitements, des classes binaires et des paramètres de votre ou vos classifieurs.


Faites attention à la prise en compte du déséquilibre que vous pourriez avoir entre vos classes, en termes de nombre d'avis dans chacune des classes. 
Vous pouvez notamment utiliser le module python `imbalanced-learn` pour tester différentes manières de gérer les déséquilibres entre classes.
	
Vous pouvez également tout d'abord considérer un sous-ensemble de votre ensemble d'apprentissage, pour mettre au point chacune de vos expérimentations.
			


## Modules python utilisés

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

## 1. Classification multiclasse avec le classifieur Arbre de décision

In [ ]:
try:
    X_train = pd.read_csv("data/train_test/x_train.csv", index_col=0)
    y_train = pd.read_csv("data/train_test/y_train.csv", index_col=0).squeeze() # squeeze pour en faire une Série
    X_test = pd.read_csv("data/train_test/x_test.csv", index_col=0)
    y_test = pd.read_csv("data/train_test/y_test.csv", index_col=0).squeeze()
    
    print(f"Données chargées. Train: {X_train.shape}, Test: {X_test.shape}")
    
    # Vérification du déséquilibre des classes
    print("\nRépartition des classes dans y_train :")
    print(y_train.value_counts(normalize=True).sort_index())
    
except FileNotFoundError:
    print("Fichiers non trouvés")


classes_noms = ['Très Négatif', 'Négatif', 'Neutre', 'Positif', 'Très Positif']

# Transformateurs pour les colonnes
preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=2000, stop_words='english'), 'text_'),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['category'])
    ],
    remainder='drop'
)

# Gestion du déséquilibre
pipeline_dt = ImbPipeline([
    ('preprocessor', preprocessor),
    ('sampler', RandomOverSampler(random_state=42)),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

# Grille de paramètres à tester
param_grid_dt = {
    'classifier__criterion': ['gini', 'entropy'],
    'classifier__max_depth': [10, 20, 30, None],
    'classifier__min_samples_split': [2, 10, 20]
}

# Recherche des meilleurs hyperparamètres
print("Démarrage de l'optimisation")
grid_dt = GridSearchCV(pipeline_dt, param_grid_dt, cv=3, scoring='accuracy', n_jobs=-1)
grid_dt.fit(X_train, y_train)

print(f"\nMeilleurs paramètres trouvés : {grid_dt.best_params_}")
print(f"Meilleur score de validation (accuracy) : {grid_dt.best_score_:.3f}")

# Évaluation sur le jeu de test avec le meilleur modèle
best_dt_model = grid_dt.best_estimator_
predictions_dt = best_dt_model.predict(X_test)

print("\nRapport de classification")a
print(classification_report(y_test, predictions_dt, target_names=classes_noms))

# Matrice de confusion
fig, ax = plt.subplots(figsize=(10, 8)) 
ConfusionMatrixDisplay.from_predictions(
    y_test, 
    predictions_dt, 
    display_labels=classes_noms,
    cmap='Blues',
    ax=ax
)
plt.title("Matrice de Confusion - Arbre de Décision")
plt.show()

## [Bonus] 2. Utilisation d'autres classifieurs multiclasse
 

## 3. Synthèse des expérimentations et des résultats obtenus